In [1]:
import pandas as pd
import yfinance as yf
import duckdb

In [3]:
ar_adrs = [
    "YPF",    # YPF S.A. (NYSE)
    "GGAL",   # Grupo Financiero Galicia (NASDAQ)
    "BMA",    # Banco Macro (NYSE)
    "BBAR",   # BBVA Argentina (NYSE)
    "PAM",    # Pampa Energia (NYSE)
    "TEO",    # Telecom Argentina (NYSE)
    "CEPU",   # Central Puerto (NYSE)
    "LOMA",   # Loma Negra (NYSE)
    "CRESY",  # Cresud (NASDAQ)
    "IRS",    # IRSA Inversiones (NYSE)
    "SUPV",   # Grupo Supervielle (NYSE)
    # "DESP",   # Despegar.com (NYSE)
    "MELI",   # MercadoLibre (NASDAQ)
    "BIOX",   # Bioceres Crop Solutions (NASDAQ)
]


In [18]:
# data = yf.download(ar_adrs, start="2018-01-01")
data = yf.download(ar_adrs, start="2018-01-01", group_by="ticker")

[*********************100%***********************]  13 of 13 completed


In [19]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2024 entries, 2018-01-02 to 2026-01-21
Data columns (total 65 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   (BIOX, Open)     1969 non-null   float64
 1   (BIOX, High)     1969 non-null   float64
 2   (BIOX, Low)      1969 non-null   float64
 3   (BIOX, Close)    1969 non-null   float64
 4   (BIOX, Volume)   1969 non-null   float64
 5   (CEPU, Open)     2002 non-null   float64
 6   (CEPU, High)     2002 non-null   float64
 7   (CEPU, Low)      2002 non-null   float64
 8   (CEPU, Close)    2002 non-null   float64
 9   (CEPU, Volume)   2002 non-null   float64
 10  (MELI, Open)     2024 non-null   float64
 11  (MELI, High)     2024 non-null   float64
 12  (MELI, Low)      2024 non-null   float64
 13  (MELI, Close)    2024 non-null   float64
 14  (MELI, Volume)   2024 non-null   int64  
 15  (BMA, Open)      2024 non-null   float64
 16  (BMA, High)      2024 non-null   float64
 

In [20]:
data.columns.levels[0]

Index(['BBAR', 'BIOX', 'BMA', 'CEPU', 'CRESY', 'GGAL', 'IRS', 'LOMA', 'MELI',
       'PAM', 'SUPV', 'TEO', 'YPF'],
      dtype='object', name='Ticker')

In [52]:
duckdb.sql('SELECT * FROM bb')

┌────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────────┬───────────────────┬─

In [45]:
bb = data.T.dropna(how='all')

In [49]:
bb = bb.T

In [50]:
bb.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2024 entries, 2018-01-02 to 2026-01-21
Data columns (total 65 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   (BIOX, Open)     1969 non-null   float64
 1   (BIOX, High)     1969 non-null   float64
 2   (BIOX, Low)      1969 non-null   float64
 3   (BIOX, Close)    1969 non-null   float64
 4   (BIOX, Volume)   1969 non-null   float64
 5   (MELI, Open)     2024 non-null   float64
 6   (MELI, High)     2024 non-null   float64
 7   (MELI, Low)      2024 non-null   float64
 8   (MELI, Close)    2024 non-null   float64
 9   (MELI, Volume)   2024 non-null   float64
 10  (GGAL, Open)     2024 non-null   float64
 11  (GGAL, High)     2024 non-null   float64
 12  (GGAL, Low)      2024 non-null   float64
 13  (GGAL, Close)    2024 non-null   float64
 14  (GGAL, Volume)   2024 non-null   float64
 15  (CEPU, Open)     2002 non-null   float64
 16  (CEPU, High)     2002 non-null   float64
 

In [53]:
def normalize_prices_long(df: pd.DataFrame) -> pd.DataFrame:
    """
    MultiIndex columns (ticker, field) -> long tidy table:
    date, ticker, open, high, low, close, adj_close, volume
    """
    out = (
        df.copy()
          .rename(columns={"Adj Close": "Adj_Close"}, level=1)
          .stack(level=0, future_stack=True)      # index: date, ticker
          .reset_index()
          .rename(columns={"level_0": "date", "level_1": "ticker"})
    )

    # standardize column names
    out.columns = [c.lower().replace(" ", "_") for c in out.columns]
    # if you renamed Adj Close -> Adj_Close above, this becomes adj_close
    return out

In [54]:
cc = normalize_prices_long(bb)

In [57]:
cc

,date,ticker,open,high,low,close,volume
0,2018-01-02,BIOX,NaN,NaN,NaN,NaN,NaN
1,2018-01-02,MELI,317.489990,322.640015,316.269989,322.579987,366900.0
2,2018-01-02,GGAL,50.807360,51.448176,50.349637,51.272713,362800.0
3,2018-01-02,CEPU,NaN,NaN,NaN,NaN,NaN
4,2018-01-02,BMA,80.831263,81.494324,80.188923,80.769104,145500.0
...,...,...,...,...,...,...,...
26307,2026-01-21,LOMA,11.690000,11.850000,11.470000,11.810000,220324.0
26308,2026-01-21,PAM,81.779999,82.050003,80.080101,81.794998,139515.0
26309,2026-01-21,CRESY,12.360000,12.700000,12.250000,12.425800,184774.0
26310,2026-01-21,IRS,16.340000,16.732000,16.200001,16.395000,73390.0


In [59]:
def build_ticker_dim(df: pd.DataFrame) -> pd.DataFrame:
    tickers = df.columns.get_level_values(0).unique()

    rows = []
    for t in tickers:
        sub = df[t]
        first_date = sub.dropna(how="all").index.min()
        last_date  = sub.dropna(how="all").index.max()
        has_data = pd.notna(first_date)

        rows.append({
            "ticker": t,
            "has_data": bool(has_data),
            "first_date": first_date,
            "last_date": last_date,
        })

    return pd.DataFrame(rows).sort_values(["has_data","ticker"], ascending=[False, True])

ticker_dim = build_ticker_dim(data)
print(ticker_dim)


   ticker  has_data first_date  last_date
13   BBAR      True 2018-01-02 2026-01-21
1    BIOX      True 2018-03-22 2026-01-21
5     BMA      True 2018-01-02 2026-01-21
4    CEPU      True 2018-02-02 2026-01-21
11  CRESY      True 2018-01-02 2026-01-21
3    GGAL      True 2018-01-02 2026-01-21
12    IRS      True 2018-01-02 2026-01-21
9    LOMA      True 2018-01-02 2026-01-21
2    MELI      True 2018-01-02 2026-01-21
10    PAM      True 2018-01-02 2026-01-21
6    SUPV      True 2018-01-02 2026-01-21
8     TEO      True 2018-01-02 2026-01-21
7     YPF      True 2018-01-02 2026-01-21
0    DESP     False        NaT        NaT


In [66]:
data.columns.levels[1]

Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')